In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [2]:
PROJECT_ROOT = Path.cwd().parent

IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "Images"
CAPTION_FILE = (PROJECT_ROOT/ "data"/ "raw"/ "annotations"/ "Flickr8k.token.txt")

print("Image directory:", IMAGE_DIR)
print("Caption file:", CAPTION_FILE)

Image directory: /home/ec2-user/SageMaker/Image Caption Genator/Image-Caption-Generator/data/raw/Images
Caption file: /home/ec2-user/SageMaker/Image Caption Genator/Image-Caption-Generator/data/raw/annotations/Flickr8k.token.txt


In [3]:
captions_df = pd.read_csv(CAPTION_FILE, sep="\t", header=None, names=["caption_id", "caption"])

print(captions_df.head())
print("Shape:", captions_df.shape)
print("Columns:", captions_df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/home/ec2-user/SageMaker/Image Caption Genator/Image-Caption-Generator/data/raw/annotations/Flickr8k.token.txt'

In [ ]:
captions_df[["image", "caption_number"]] = (captions_df["caption_id"].str.rsplit("#", n=1, expand=True))

captions_df["caption_number"] = captions_df["caption_number"].astype(int)

captions_df = captions_df[["image", "caption_number", "caption"]]

print(captions_df.head())

In [ ]:
image_files = list(IMAGE_DIR.glob("*.jpg"))

print("Number of image files:", len(image_files))
print("Number of unique captioned images:", captions_df["image"].nunique())
print("Missing captions:", captions_df["caption"].isna().sum())

In [ ]:
captions_per_image = captions_df.groupby("image").size()

print(captions_per_image.describe())
print(captions_per_image.value_counts())

In [ ]:
sample_image_name = captions_df["image"].iloc[0]
sample_image_path = IMAGE_DIR / sample_image_name

image = Image.open(sample_image_path)

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.show()

sample_captions = captions_df.loc[
    captions_df["image"] == sample_image_name,
    "caption"
]

for number, caption in enumerate(sample_captions, start=1):
    print(f"{number}. {caption}")

In [ ]:
image_names = {path.name for path in IMAGE_DIR.glob("*.jpg")}
captioned_image_names = set(captions_df["image"])

missing_images = sorted(captioned_image_names - image_names)
extra_images = sorted(image_names - captioned_image_names)

print("Captioned images missing from folder:", missing_images)
print("Images without captions:", extra_images)

In [ ]:
ANNOTATION_DIR = PROJECT_ROOT / "data" / "raw" / "annotations"

split_files = {
    "train": "Flickr_8k.trainImages.txt",
    "validation": "Flickr_8k.devImages.txt",
    "test": "Flickr_8k.testImages.txt",
}

for split_name, filename in split_files.items():
    split_images = set(
        pd.read_csv(
            ANNOTATION_DIR / filename,
            header=None
        )[0].str.strip()
    )

    overlap = split_images.intersection(missing_images)
    print(f"{split_name}: {overlap}")

In [ ]:
captions_df = captions_df[
    ~captions_df["image"].isin(missing_images)
].reset_index(drop=True)

In [15]:
print("Caption rows after filtering:", len(captions_df))
print("Unique usable images:", captions_df["image"].nunique())

Caption rows after filtering: 40455
Unique usable images: 8091
